# SankhyaVox — HMM (GMM-HMM) Classifier Training

Train and evaluate the GMM-HMM baseline on augmented data, test on held-out human speaker.

**Prerequisites:** Run `DataPipeline().build()` and augmentation first.

In [ ]:
# ── stdlib ──
import pickle
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple

# ── third-party ──
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Config

In [ ]:
# ── Paths ──
DATA_PROCESSED = "/kaggle/input/datasets/devarabhavana/sankhyavox-dataset/data_processed"
PROCESSED_DIR  = DATA_PROCESSED          # alias used by SankhyaVoxDataset default
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Human speaker to hold out for testing
TEST_SPEAKER = "S05"

# ── HMM parameters ──
# State count per token — sized to phonetic complexity
HMM_STATES = {
    "shunya":  5,
    "eka":     4,   # bumped from 3 — was confused with tri
    "dvi":     4,   # bumped from 3 — was confused with tri
    "tri":     3,
    "catur":   5,
    "pancha":  5,
    "shat":    4,   # bumped from 3 — was confused with shata
    "sapta":   5,
    "ashta":   5,
    "nava":    4,
    "dasha":   4,
    "vimsati": 6,
    "shata":   4,
}

# Per-token GMM mixtures — more for confused tokens
GMM_MIXTURES = {
    "shunya":  2,
    "eka":     4,   # confused with tri/pancha — more mixtures
    "dvi":     4,   # confused with tri — more mixtures
    "tri":     3,
    "catur":   2,
    "pancha":  4,   # confused with eka — more mixtures
    "shat":    4,   # confused with shata — more mixtures
    "sapta":   2,
    "ashta":   2,
    "nava":    2,
    "dasha":   2,
    "vimsati": 2,
    "shata":   4,   # confused with shat — more mixtures
}

BAUM_WELCH_ITERS   = 150       # EM iterations for training

# Some other config
# ═══════════════════════════════════════════════════════════════════════════════
# VOCABULARY
# The 13 base Sanskrit tokens that compose all numbers 0–99
# ═══════════════════════════════════════════════════════════════════════════════

VOCAB = [
    "shunya",    # 0
    "eka",       # 1
    "dvi",       # 2
    "tri",       # 3
    "catur",     # 4
    "pancha",    # 5
    "shat",      # 6
    "sapta",     # 7
    "ashta",     # 8
    "nava",      # 9
    "dasha",     # 10
    "vimsati",   # 20
    "shata",     # 100
]

TOKEN_TO_IDX = {tok: i for i, tok in enumerate(VOCAB)}
IDX_TO_TOKEN = {i: tok for i, tok in enumerate(VOCAB)}

# ═══════════════════════════════════════════════════════════════════════════════
# NUMERIC ID MAPPING
# Zero-padded 3-digit IDs used in the processed file naming convention:
#   <SpeakerId>_<numericId>_<rep>.wav   e.g. S01_001_03.wav
# ═══════════════════════════════════════════════════════════════════════════════

TOKEN_TO_NUMERIC_ID = {
    "shunya":  "000",
    "eka":     "001",
    "dvi":     "002",
    "tri":     "003",
    "catur":   "004",
    "pancha":  "005",
    "shat":    "006",
    "sapta":   "007",
    "ashta":   "008",
    "nava":    "009",
    "dasha":   "010",
    "vimsati": "020",
    "shata":   "100",
}
NUMERIC_ID_TO_TOKEN = {v: k for k, v in TOKEN_TO_NUMERIC_ID.items()}

# Numeric value each token represents (used as classification labels)
TOKEN_TO_VALUE = {
    "shunya": 0, "eka": 1, "dvi": 2, "tri": 3, "catur": 4,
    "pancha": 5, "shat": 6, "sapta": 7, "ashta": 8, "nava": 9,
    "dasha": 10, "vimsati": 20, "shata": 100,
}
VALUE_TO_TOKEN = {v: k for k, v in TOKEN_TO_VALUE.items()}

## SankhyaVoxDataset

**Paste the `SankhyaVoxDataset` class from `dataset/dataset.py` below.**

In [ ]:
# ╔═════════════════════════════════════════════════════════════════════╗
# ║  PASTE SankhyaVoxDataset class from dataset/dataset.py HERE         ║
# ╚═════════════════════════════════════════════════════════════════════╝

## Load Data & Split

In [ ]:
# Training: augmented data, excluding held-out speaker
aug_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["augmented"])
print(f"Full augmented: {repr(aug_ds)}")
print(f"Augmented speakers: {aug_ds.speakers}")

# Exclude the test speaker's augmented variant
aug_test_speaker = f"aug{TEST_SPEAKER}"  # e.g. "augS05"
train_ds = aug_ds.exclude_speakers([aug_test_speaker])
X_train, y_train = train_ds.get_Xy()
print(f"\nTrain: {len(X_train)} samples (excluded {aug_test_speaker})")

# Testing: real human recordings for held-out speaker
human_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["human"])
test_ds = human_ds.filter(speaker=TEST_SPEAKER)
X_test, y_test = test_ds.get_Xy()
print(f"Test:  {len(X_test)} samples (human {TEST_SPEAKER})")
print(f"Labels in test: {sorted(set(y_test))}")
print(f"Feature shape example: {X_train[0].shape}")

print("=" * 70)
print("TRAIN DATA SUMMARY:")
print("=" * 70)
print(train_ds.summary())
print()

print("=" * 70)
print("TEST DATA SUMMARY:")
print("=" * 70)
print(test_ds.summary())
print()

---
## HMM Classifier

**Paste the `SankhyaHMM` class from `models/hmm_classifier.py` below.**

In [ ]:
# !pip install hmmlearn
from hmmlearn.hmm import GMMHMM

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PASTE SankhyaHMM class from models/hmm_classifier.py HERE ║
# ╚══════════════════════════════════════════════════════════════╝

## Train

In [ ]:
hmm = SankhyaHMM(n_iter=BAUM_WELCH_ITERS, mix_map=GMM_MIXTURES, states_map=HMM_STATES)
hmm.fit(X_train, y_train)
print("Training complete.")

## Save

In [ ]:
hmm.save(str(CHECKPOINT_DIR / "hmm_classifier.pkl"))

## Load

In [ ]:
hmm_loaded = SankhyaHMM(checkpoint_path=str(CHECKPOINT_DIR / "hmm_classifier.pkl"))

## Test

In [ ]:
hmm_preds = hmm_loaded.predict(X_test)
hmm_acc = accuracy_score(y_test, hmm_preds)
print(f"HMM Accuracy: {hmm_acc:.3f}")
print()
print(classification_report(y_test, hmm_preds, zero_division=0))

## Results

In [ ]:
# Token label mapping for display
VALUE_TO_TOKEN = {
    0: "shunya", 1: "eka", 2: "dvi", 3: "tri", 4: "catur",
    5: "pancha", 6: "shat", 7: "sapta", 8: "ashta", 9: "nava",
    10: "dasha", 20: "vimsati", 100: "shata",
}

labels_sorted = sorted(set(y_test))
label_names = [VALUE_TO_TOKEN.get(l, str(l)) for l in labels_sorted]

cm = confusion_matrix(y_test, hmm_preds, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=label_names, yticklabels=label_names)
ax.set_title(f"HMM Confusion Matrix \u2014 Accuracy: {hmm_acc:.3f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, l in enumerate(labels_sorted):
    row_total = cm[i].sum()
    correct = cm[i, i]
    acc = correct / row_total if row_total > 0 else 0
    print(f"  {VALUE_TO_TOKEN.get(l, str(l)):>8s} ({l:>3d}): {correct}/{row_total} = {acc:.2%}")